# Personal Information
Name: **Tejaswi Madduri**

StudentID: **16233840**

Email: [**tejaswi.madduri@student.uva.nl**](tejaswi.madduri@student.uva.nl)

Submitted on: **23.03.2026**

# Data Context
For my thesis on analyzing energy consumption and resource utilization in cloud environments, I use datasets from the IFCA Cloud Energy Consumption project. The objective of this study is to understand how system performance metrics such as CPU usage, memory utilization, and network activity relate to energy consumption within cloud infrastructure.

The data consists of system-level monitoring information, hardware configuration details, and virtual machine specifications collected from a cloud computing environment. It includes time-series measurements of resource usage and power consumption, as well as metadata describing the underlying infrastructure and virtual machines. Together, these datasets provide a comprehensive view of how computational resources are utilized and how energy is consumed, enabling further analysis of performance and efficiency in cloud systems.

# Data Description

The analysis is based on time-series data from a cloud computing environment, including system-level metrics such as CPU usage, memory utilization, and power consumption across multiple nodes. The data enables the study of energy behavior over time and across different machines.

Initial exploration shows that CPU usage is highly skewed toward low values, while power consumption remains relatively stable, indicating baseline energy usage. Energy spikes are defined as the top 5% of power values, resulting in a highly imbalanced dataset. Spatial analysis identifies certain nodes as persistent hotspots with consistently higher energy consumption.

Overall, the dataset contains skewed distributions, missing values, and weak correlations between features, highlighting the need for preprocessing and interpretable modelling approaches.

In [ ]:
#Imports
import pandas as pd
import glob


### Data Loading

In [ ]:
groups = pd.read_csv("node-groups/2024-12-14T000000Z_2025-04-13T235959Z/groups.csv")


node_files = glob.glob("nodes/2024-12-14T000000Z_2025-04-13T235959Z/*/*.csv")
nodes = pd.concat([
    pd.read_csv(f).assign(node_name=f.split('/')[-2])
    for f in node_files
], ignore_index=True)


vms = pd.read_csv("vms/2024-12-14T000000Z_2025-04-13T235959Z/vms.csv")


node_vm_files = glob.glob("nodes-vms/2024-12-14T000000Z_2025-04-13T235959Z/*/*/*.csv")
node_vms = pd.concat([pd.read_csv(f) for f in node_vm_files], ignore_index=True)

nodes.head()

All datasets (nodes, groups, VMs, and node-VM mappings) are loaded and combined to enable integrated analysis.

### Basic Overview

In [ ]:
nodes.shape
nodes.info()

The dataset contains time-series system metrics across multiple nodes with mostly numerical features.

### Timestamp Processing 

In [ ]:
nodes['timestamp'] = pd.to_datetime(nodes['timestamp'])
nodes = nodes.sort_values('timestamp')

Timestamps are converted to datetime format to support temporal analysis.

### Power Over Time (Temporal Analysis)

In [ ]:
nodes.set_index('timestamp')['ipmi_system_power_watts'].plot()

Power consumption varies over time, showing fluctuations and potential spike events.

### Spike Detection

In [ ]:
threshold = nodes['ipmi_system_power_watts'].quantile(0.95)

nodes['spike'] = (nodes['ipmi_system_power_watts'] > threshold).astype(int)

nodes['spike'].value_counts()

Energy spikes are defined as the top 5% of power values, resulting in an imbalanced dataset.

### Hotspot Detection (Spatial)

In [ ]:
nodes.groupby('node_name')['ipmi_system_power_watts'].mean().sort_values(ascending=False).head()

Nodes with consistently high average power consumption are identified as hotspots.

### CPU vs Power Relationship

In [ ]:
import seaborn as sns

sns.scatterplot(
    x=nodes['cpu_usage_percent'],
    y=nodes['ipmi_system_power_watts']
)

Power consumption is not strongly dependent on CPU usage, indicating baseline energy behavior.

### Correlation Analysis

In [ ]:
nodes[['cpu_usage_percent','memory_used_bytes',
       'ipmi_system_power_watts']].corr()

Weak correlations suggest that multiple factors influence energy consumption.

### Missing Values

In [ ]:
nodes.isnull().sum().sort_values(ascending=False).head()

Missing values are present in several features and must be handled before modelling.

### Data Cleaning

In [ ]:
nodes_clean = nodes.dropna(subset=['cpu_usage_percent',
                                  'memory_used_bytes',
                                  'ipmi_system_power_watts'])

Rows with missing values in key features are removed to ensure reliable analysis.

### Hardware Analysis (Groups)

In [ ]:
merged_groups = nodes_clean.merge(groups, on='node_group', how='left')

merged_groups.groupby('node_group')['ipmi_system_power_watts'].mean()

Different hardware configurations show variations in energy consumption.

### VM Analysis

In [ ]:
vms[['vcpus','memory_mb']].describe()

Virtual machines vary in size, indicating different workload intensities.

### Node-VM Relationship

# align column name
node_vms = node_vms.rename(columns={'hypervisor_name': 'node_name'})

# count VMs per node
vm_counts = node_vms.groupby('node_name').size().reset_index(name='vm_count')

# merge
merged_vm = nodes_clean.merge(vm_counts, on='node_name', how='left')

# relationship
sns.scatterplot(
    x=merged_vm['vm_count'],
    y=merged_vm['ipmi_system_power_watts']
)

Nodes hosting more virtual machines tend to consume more energy, indicating workload impact.

### Prepare Data for Model

In [ ]:
model_data = nodes_clean[['cpu_usage_percent',
                          'memory_used_bytes',
                          'spike']].dropna()

Relevant features are selected and cleaned for modelling.

### Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = model_data[['cpu_usage_percent','memory_used_bytes']]
y = model_data['spike']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

The dataset is split to evaluate model performance on unseen data.

### Baseline Model

A Logistic Regression model is used as a baseline to detect energy spikes based on system metrics. Due to class imbalance, class weighting is applied to improve detection of spike events. While the model provides a reference performance, it highlights the limitations of simple approaches and the need for more advanced methods.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))